# 15 — Walk-Forward ML Challenger vs Baseline Robot vs BIST100

Bu notebook ayrı ve dürüst bir ML karşılaştırması üretir.

**Neden başlangıç 2025?**

`Big_Winner_Label_2R`, Logistic Regression ve Q40 filtresi 2018–2024
araştırmasıyla seçildi. Bunları 2018'e geri uygulayıp tam dönem grafik
oluşturmak seçim yanlılığı yaratır. Bu nedenle karşılaştırma 2025'te başlar.

**Walk-forward akışı**

- Her ayın başında model yeniden eğitilir.
- Yalnızca o tarihten önce sinyali oluşmuş ve 5 günlük embargo öncesinde
  kapanmış işlemler eğitime girer.
- Model tipi ve Q40 eşiği değiştirilmez.
- Ay içindeki günlük Robot AL satırları yeni modelle skorlanır.
- Baseline Robot, ML Challenger ve BIST100 aynı tarihlerde karşılaştırılır.

2025+ sonuçları daha önce incelendiği için kusursuz bir holdout değildir;
ancak tahmin üretiminde gelecekteki işlem sonucu kullanılmaz.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import (
    BASE_FEATURE_COLUMNS,
    add_meta_features,
)
from src.ml_targets import add_alternative_targets
from src.dual_paper import (
    load_challenger_deployment,
    deployment_summary,
)
from src.ml_walkforward import (
    WalkForwardConfig,
    generate_walk_forward_probabilities,
    evaluate_walk_forward_comparison,
    save_walk_forward_artifacts,
)


## 1. Veri ve kilitli ML kararını yükle


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

events = pd.read_parquet(
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)

for column in [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]:
    events[column] = pd.to_datetime(
        events[column]
    )

events = add_alternative_targets(events)

deployment, _ = load_challenger_deployment(
    PROJECT_ROOT
)

display(deployment_summary(deployment))

if deployment.target != "Big_Winner_Label_2R":
    raise RuntimeError(
        "Beklenen hedef Big_Winner_Label_2R değil."
    )

if deployment.probability_threshold is None:
    raise RuntimeError(
        "Kilitli olasılık eşiği bulunamadı."
    )


## 2. Robot ve ML özelliklerini hazırla


In [ ]:
stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(
    market_features
)

scored_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=False,
)

featured_prices = add_meta_features(
    scored_prices=scored_prices,
    market_features=market_features,
)

WALK_FORWARD_START = "2025-01-01"
WALK_FORWARD_END = min(
    featured_prices["Date"].max(),
    market_prices["Date"].max(),
).strftime("%Y-%m-%d")

print(
    "Walk-forward dönemi:",
    WALK_FORWARD_START,
    "→",
    WALK_FORWARD_END,
)


## 3. Aylık walk-forward olasılıkları üret


In [ ]:
walk_forward_config = WalkForwardConfig(
    start=WALK_FORWARD_START,
    end=WALK_FORWARD_END,
    retrain_frequency="MS",
    embargo_days=5,
    minimum_training_events=500,
    random_state=42,
)

probability_prices, training_log = (
    generate_walk_forward_probabilities(
        featured_prices=featured_prices,
        events=events,
        model_name=deployment.model_name,
        target_column=deployment.target,
        feature_columns=BASE_FEATURE_COLUMNS,
        config=walk_forward_config,
    )
)

display(training_log)

print(
    "Minimum skor kapsaması:",
    training_log["Score_Coverage_%"].min(),
)


## 4. Üç portföyü aynı dönemde karşılaştır


In [ ]:
(
    comparison_metrics,
    equity_comparison,
    yearly_table,
    active_table,
    walk_forward_outputs,
) = evaluate_walk_forward_comparison(
    probability_prices=probability_prices,
    market_prices=market_prices,
    probability_threshold=(
        deployment.probability_threshold
    ),
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    start=WALK_FORWARD_START,
    end=WALK_FORWARD_END,
)

display(comparison_metrics)
display(active_table)
display(yearly_table)


## 5. Portföy değer eğrisi


In [ ]:
import matplotlib.pyplot as plt

chart = equity_comparison.set_index("Date")

plt.figure(figsize=(13, 7))
plt.plot(
    chart.index,
    chart["Baseline_Robot"],
    label="Baseline Robot",
)
plt.plot(
    chart.index,
    chart["ML_Challenger"],
    label="ML Challenger",
)
plt.plot(
    chart.index,
    chart["BIST100_Gross"],
    label="BIST100 Gross",
)
plt.title(
    "Walk-Forward: Baseline Robot, ML Challenger ve BIST100"
)
plt.xlabel("Tarih")
plt.ylabel("Portföy Değeri (TL)")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Sonuçları dashboard için kaydet


In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
    / "walk_forward"
)

metadata = {
    "method": "Monthly expanding walk-forward",
    "start": WALK_FORWARD_START,
    "end": WALK_FORWARD_END,
    "retrain_frequency": (
        walk_forward_config.retrain_frequency
    ),
    "embargo_days": (
        walk_forward_config.embargo_days
    ),
    "target": deployment.target,
    "model": deployment.model_name,
    "filter": deployment.filter_name,
    "probability_threshold": (
        deployment.probability_threshold
    ),
    "feature_count": len(
        BASE_FEATURE_COLUMNS
    ),
    "selection_window": "2018-2024",
    "evaluation_window": "2025+",
    "warning": (
        "2025+ was previously inspected; this is a "
        "leakage-controlled audit, not a pristine holdout."
    ),
}

artifact_paths = save_walk_forward_artifacts(
    output_directory=OUTPUT_DIR,
    comparison_metrics=comparison_metrics,
    equity_comparison=equity_comparison,
    yearly_table=yearly_table,
    active_table=active_table,
    training_log=training_log,
    probability_prices=probability_prices,
    metadata=metadata,
)

for name, path in artifact_paths.items():
    print(name, "→", path)


In [ ]:
import numpy as np
audit_features = featured_prices.loc[
    featured_prices["Date"].between(
        "2025-01-01",
        "2025-02-28",
    )
    & featured_prices["Signal"].eq("AL"),
    ["Date", "Ticker", *BASE_FEATURE_COLUMNS],
].copy()

records = []

for month, group in audit_features.groupby(
    audit_features["Date"].dt.to_period("M")
):
    missing_counts = (
        group[BASE_FEATURE_COLUMNS]
        .replace([np.inf, -np.inf], np.nan)
        .isna()
        .sum()
    )

    for feature, count in missing_counts[
        missing_counts.gt(0)
    ].items():
        records.append(
            {
                "Month": str(month),
                "Feature": feature,
                "Missing_Count": int(count),
                "Event_Count": len(group),
                "Missing_Rate_%": (
                    count / len(group) * 100
                ),
            }
        )

missing_feature_audit = (
    pd.DataFrame(records)
    .sort_values(
        ["Month", "Missing_Count"],
        ascending=[True, False],
    )
)

display(missing_feature_audit)

In [ ]:
market_diagnostics = pd.DataFrame(
    [
        {
            "Dataset": "market_prices",
            "Start": market_prices["Date"].min(),
            "End": market_prices["Date"].max(),
            "Rows": len(market_prices),
        },
        {
            "Dataset": "market_features",
            "Start": market_features["Date"].min(),
            "End": market_features["Date"].max(),
            "Rows": len(market_features),
        },
        {
            "Dataset": "featured_prices MARKET_RET_63",
            "Start": featured_prices.loc[
                featured_prices["MARKET_RET_63"].notna(),
                "Date",
            ].min(),
            "End": featured_prices.loc[
                featured_prices["MARKET_RET_63"].notna(),
                "Date",
            ].max(),
            "Rows": featured_prices[
                "MARKET_RET_63"
            ].notna().sum(),
        },
    ]
)

display(market_diagnostics)